# 05. Validation 오류 분석(E06)

**질문**: 데드리프트(7)·바벨 로우(9)·덤벨 로우(2)는 왜 헷갈리는가? "사람이 작아서"인가, "그 순간의 자세가 원래 같아서"인가?

- 분석 대상은 **Validation만** 씁니다(Test는 열지 않습니다).
- 주 모델은 E04 Full FT(기본 조건 중 최고, Val Macro F1 0.858)입니다. 비교용으로 E01 Scratch를 함께 봅니다.
- 바벨·덤벨 묶음(2·7·8·9)의 Val은 **모두 Z23 한 사람**입니다. 여기서 보는 패턴은 우선 "이 사람에 대한 관찰"입니다. Train과 비교해서 데이터 전체의 성질인지 따로 확인합니다.
- 원본 이미지가 들어간 그림은 `figures/samples/`(Git 제외)에 저장합니다.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import f1_score

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from common import DATA_RAW, FIGURES_SAMPLES, class_names, set_korean_font
from dataset import load_split
from predict import predict_split

print("font:", set_korean_font())
pd.set_option("display.width", 180)
NAMES = class_names()
FIGURES_SAMPLES.mkdir(parents=True, exist_ok=True)

font: Malgun Gothic


## 1. 예측 표 만들기와 검산

저장한 최고 시점 가중치(`models/E04_full.pt`, `models/E01_scratch.pt`)로 Val을 다시 예측합니다. 학습 때 기록한 Macro F1과 같아야 합니다(가중치 저장·불러오기 검산).

In [2]:
PRED = {}
for rid, s in (("E04", "full"), ("E01", "scratch")):
    d = predict_split(rid, s)
    logged = json.loads((ROOT / "experiments" / rid / s / "summary.json").read_text(encoding="utf-8"))["val"]["macro_f1"]
    print(f"{rid} {s}: 다시 계산한 Macro F1 {d.attrs['macro_f1']:.4f} / 학습 때 기록 {logged:.4f}")
    PRED[rid] = d
BB = [2, 7, 8, 9]

E04 full: 다시 계산한 Macro F1 0.8581 / 학습 때 기록 0.8581


E01 scratch: 다시 계산한 Macro F1 0.7928 / 학습 때 기록 0.7928


## 2. 오답은 "클래스 × 카메라" 칸에 몰립니다

카메라 A~E는 촬영 공간에 고정된 다섯 방향입니다. 아래는 바벨·덤벨 묶음의 클래스별·카메라별 정답률입니다.

In [3]:
for rid, d in PRED.items():
    b = d[d.target_id.isin(BB)]
    t = b.pivot_table(index="target_id", columns="camera", values="correct", aggfunc="mean").round(2)
    t.index = [f"{c} {NAMES[c]}" for c in t.index]
    print(f"[{rid}] 정답률"); display(t)
print("카메라별 전체 정답률(E04):", PRED["E04"].groupby("camera").correct.mean().round(3).to_dict())

[E04] 정답률


camera,A,B,C,D,E
2 덤벨 벤트오버 로우,0.72,0.69,1.00,0.85,0.73
7 바벨 데드리프트,0.26,0.24,0.10,0.64,0.69
8 바벨 런지,0.94,0.94,0.85,1.00,1.00
9 바벨 로우,0.80,1.00,0.50,0.00,0.17


[E01] 정답률


camera,A,B,C,D,E
2 덤벨 벤트오버 로우,0.91,0.81,0.90,0.00,0.00
7 바벨 데드리프트,0.00,0.00,0.69,0.76,0.74
8 바벨 런지,0.89,0.66,0.19,0.96,0.97
9 바벨 로우,0.87,0.00,0.00,0.73,0.92


카메라별 전체 정답률(E04): {'A': 0.837, 'B': 0.866, 'C': 0.861, 'D': 0.887, 'E': 0.88}


In [4]:
tr = load_split("train")
t = tr[tr.target_id.isin(BB)].pivot_table(index="target_id", columns="camera", values="image_id", aggfunc="size")
t.index = [f"{c} {NAMES[c]}" for c in t.index]
print("Train 장수(클래스 × 카메라): 카메라별로 고르게 들어 있다"); t

Train 장수(클래스 × 카메라): 카메라별로 고르게 들어 있다


camera,A,B,C,D,E
2 덤벨 벤트오버 로우,159,158,161,162,160
7 바벨 데드리프트,160,160,167,159,154
8 바벨 런지,162,159,159,159,161
9 바벨 로우,67,66,59,57,63


- **카메라 하나가 전체적으로 나쁜 것은 아닙니다**(E04 카메라별 전체 정답률 0.84~0.89).
- 대신 특정 칸이 무너집니다.
  - E04: 데드리프트 카메라 A·B·C(0.10~0.26), 바벨 로우 카메라 D·E(0.00·0.17)
  - E01: 덤벨 로우 D·E, 데드리프트 A·B, 바벨 로우 B·C가 모두 0.00
- Train에는 네 클래스가 카메라 다섯 대에 고르게 들어 있습니다(바벨 로우만 전체적으로 적음). 그러니 "그 방향 사진을 Train에서 못 봐서"가 아닙니다. **같은 자세가 어떤 방향에서는 구분되고 어떤 방향에서는 구분되지 않는다**는 뜻입니다.
- 모델마다 무너지는 칸이 다릅니다. 모델이 두 운동을 가르는 확실한 단서를 잡지 못하고, 방향마다 다른 우연한 단서에 기대고 있다는 신호입니다.

## 3. 카메라별로 직접 보기(Z23, E04 예측)

제목의 초록은 맞힘, 빨강은 틀림입니다. 괄호는 그 칸의 정답률입니다.

In [5]:
d = PRED["E04"].copy()
fig, ax = plt.subplots(3, 5, figsize=(20, 7.5))
for i, c in enumerate((2, 7, 9)):
    for j, cam in enumerate("ABCDE"):
        sub = d[(d.target_id == c) & (d.camera == cam)].sort_values("frame_no")
        r = sub.iloc[len(sub) // 2]
        ax[i, j].imshow(Image.open(DATA_RAW / r.image_path)); ax[i, j].axis("off")
        ax[i, j].set_title(f"{c} {NAMES[c]} cam{cam} f{r.frame_no}\n예측 {r.pred} ({sub.correct.mean():.0%} 정답)", fontsize=9,
                           color="green" if r.correct else "red")
plt.tight_layout(); plt.savefig(FIGURES_SAMPLES / "e06_val_z23_by_camera.jpg", dpi=65); plt.show()

[원본 사진 그림 출력은 지웠습니다. 로컬 파일 figures/samples/에 저장되어 있고, 공개 저장소에는 넣지 않습니다(AI Hub 이용 조건).]


In [6]:
def grid(d, cls, title, per_cam=None, fname=None):
    s = d[d.target_id == cls].sort_values(["camera", "clip_id", "frame_no"])
    if per_cam: s = s.groupby("camera").head(per_cam)
    cols = 8; rows = -(-len(s) // cols)
    fig, ax = plt.subplots(rows, cols, figsize=(cols * 2.1, rows * 2.6))
    for a in ax.ravel(): a.axis("off")
    for a, (_, r) in zip(ax.ravel(), s.iterrows()):
        a.imshow(Image.open(DATA_RAW / r.image_path).crop((96, 20, 288, 216)))  # 가운데를 잘라 사람 확대
        a.set_title(f"cam{r.camera} f{r.frame_no} →{r.pred} {r.conf:.2f}", fontsize=8, color="green" if r.correct else "red")
    fig.suptitle(title); plt.tight_layout()
    if fname: plt.savefig(FIGURES_SAMPLES / fname, dpi=60)
    plt.show()
grid(d, 9, "Val 바벨 로우(9) 64장 전부, E04 예측(→예측 클래스, 확신도)", fname="e06_val_row_all.jpg")

[원본 사진 그림 출력은 지웠습니다. 로컬 파일 figures/samples/에 저장되어 있고, 공개 저장소에는 넣지 않습니다(AI Hub 이용 조건).]


In [7]:
grid(d, 7, "Val 데드리프트(7), 카메라별 8장씩, E04 예측", per_cam=8, fname="e06_val_deadlift_by_cam.jpg")

[원본 사진 그림 출력은 지웠습니다. 로컬 파일 figures/samples/에 저장되어 있고, 공개 저장소에는 넣지 않습니다(AI Hub 이용 조건).]


### 3.1 사진에서 보이는 것(헬스 관점)

- **데드리프트**: Z23의 데드리프트 사진은 대부분 *바벨을 허벅지 앞에 들고 똑바로 선 자세*(다 든 순간, lockout)입니다.
  - 카메라 A·B(뒤쪽)와 C(정면)에서는 이 선 자세가 거의 바벨 로우로 예측됩니다.
  - 카메라 D·E(옆쪽)에서는 같은 선 자세도 대부분 맞힙니다. 옆에서 보면 바벨이 허벅지에 붙어 있고 상체가 완전히 펴진 것이 보입니다. 정면·뒤에서는 상체 각도가 잘 드러나지 않습니다.
  - 상체를 숙인(바벨이 바닥 쪽인) 프레임은 비교적 잘 맞힙니다.
- **바벨 로우**: 대부분 상체를 숙인 전형적인 자세이고, 카메라 A·B에서는 잘 맞힙니다.
  - 틀린 사진에는 *똑바로 선 프레임*(데드리프트 lockout과 같은 모습)이 있습니다. 또 *바벨을 가슴이나 머리 높이로 든 특이한 프레임*도 많습니다.
  - 카메라 D에서는 덤벨 로우(2)로, E에서는 데드리프트(7)로 많이 틀립니다. 화면 속 바벨은 몇 픽셀 두께의 가는 선이라, 줄인 입력에서는 "바벨인지 덤벨인지"를 보기 어렵습니다(가설).
- 바벨 로우와 데드리프트는 **둘 다 바벨을 잡고 엉덩이를 접는(힙 힌지) 동작**입니다. 로우는 숙인 자세를 유지한 채 바벨을 몸 쪽으로 당기고, 데드리프트는 바닥에서 들어 올려 끝까지 일어섭니다. 차이는 "어느 자세에 머무는가"와 "움직임의 방향"인데, 사진 한 장에는 움직임이 없습니다.

## 4. Train에서도 같은가(Z23만의 문제인지)

Train의 바벨 로우·데드리프트를 사람마다 8장씩 뽑아 봅니다(Train은 분석해도 됩니다).

In [8]:
for cls in (9, 7):
    c = tr[tr.target_id == cls]
    s = pd.concat([c[c.person_id == p].sample(8, random_state=42) for p in sorted(c.person_id.unique())]).sort_values(["person_id", "camera"])
    fig, ax = plt.subplots(5, 8, figsize=(16.8, 13))
    for a in ax.ravel(): a.axis("off")
    for a, (_, r) in zip(ax.ravel(), s.iterrows()):
        a.imshow(Image.open(DATA_RAW / r.image_path).crop((96, 20, 288, 216))); a.set_title(f"{r.person_id} cam{r.camera} f{r.frame_no}", fontsize=8)
    fig.suptitle(f"Train {cls} {NAMES[cls]}: 사람별 8장"); plt.tight_layout()
    plt.savefig(FIGURES_SAMPLES / f"e06_train_{'row' if cls == 9 else 'deadlift'}_sample.jpg", dpi=55); plt.show()

[원본 사진 그림 출력은 지웠습니다. 로컬 파일 figures/samples/에 저장되어 있고, 공개 저장소에는 넣지 않습니다(AI Hub 이용 조건).]


- Train의 바벨 로우에도 **똑바로 선 프레임**이 섞여 있습니다(예: Z17 cam D f21, Z18 cam B f21, Z19 cam D f17). 데드리프트에도 선 자세와 숙인 자세가 섞여 있습니다. 그러니 "선 자세에서 두 운동이 같아 보인다"는 Z23만의 문제가 아니라 **데이터 자체의 성질**입니다(01_eda의 가설 H-e와 같음).
- 반면 Z23 바벨 로우의 *바벨을 가슴·머리 높이로 든 자세*는 이 Train 표본 40장에서는 거의 보이지 않았습니다. Val 사람 고유의 동작(또는 자세 변형 상태)일 수 있습니다. 표본이라 "Train에 없다"고 단정하지는 않습니다.

## 5. 오답의 확신도와 클립 단위로 본 오류

- **확신도**: 틀린 예측에 얼마나 자신 있었는가
- **클립 평균(분석용)**: 같은 클립 8장(카메라 2대 × 시점 4개)의 확률을 평균해 클립 하나에 하나의 예측을 내리면 어떻게 되는가. 이 프로젝트의 문제는 *사진 한 장* 분류라서, 이것은 모델 성능이 아니라 **오류가 프레임 하나에만 있는지, 클립 전체에 있는지**를 보는 진단입니다.

In [9]:
rows = []
for rid, d in PRED.items():
    b = d[d.target_id.isin(BB)]; w = b[~b.correct]
    pcols = [f"p{c}" for c in range(12)]
    clip_pred = d.groupby("clip_id")[pcols].transform("mean").values.argmax(1)
    row = {"모델": rid, "오답 확신도 중앙값": round(w.conf.median(), 3), "정답 확신도 중앙값": round(b[b.correct].conf.median(), 3),
           "확신도>0.9 오답 비율": round((w.conf > 0.9).mean(), 3),
           "이미지 Macro F1": round(f1_score(d.target_id, d.pred, average="macro"), 4),
           "클립 평균 Macro F1(분석용)": round(f1_score(d.target_id, clip_pred, average="macro"), 4)}
    for c in (2, 7, 9):
        m = (d.target_id == c).values
        row[f"{c} 이미지→클립 정답률"] = f"{d.correct[m].mean():.2f}→{(clip_pred[m] == c).mean():.2f}"
    rows.append(row)
pd.DataFrame(rows)

,모델,오답 확신도 중앙값,정답 확신도 중앙값,확신도>0.9 오답 비율,이미지 Macro F1,클립 평균 Macro F1(분석용),2 이미지→클립 정답률,7 이미지→클립 정답률,9 이미지→클립 정답률
0,E04,0.826,0.992,0.359,0.8581,0.9082,0.79→1.00,0.40→0.35,0.53→0.88
1,E01,0.728,0.960,0.285,0.7928,0.9260,0.51→0.85,0.44→0.50,0.50→0.88


- **틀릴 때도 자신 있게 틀립니다.** E04 오답의 36%는 확신도가 0.9를 넘습니다. 03 노트북에서 본 "Val Loss 상승(과신)"과 같은 이야기입니다.
- **바벨 로우·덤벨 로우의 오답은 프레임 일부의 문제입니다.** 클립 8장을 평균하면 정답률이 크게 오릅니다(E04 바벨 로우 0.53 → 0.88, 덤벨 로우 0.79 → 1.00). 한 클립 안에 전형적인 숙인 자세가 충분히 있어서, 애매한 프레임 몇 장을 덮어 줍니다.
- **데드리프트는 클립 단위로도 풀리지 않습니다**(E04 0.40 → 0.35). Z23의 데드리프트 클립은 선 자세 프레임이 대부분이라, 클립 전체가 로우로 읽힙니다.

## 6. 결론: 원인 정리와 다음 실험(E07) 후보

| 혼동 | 주된 원인(관찰 기반) | 해상도·crop으로 나아질까(H-b) |
|---|---|---|
| 데드리프트 → 바벨 로우 | 선 자세(lockout)는 정면·뒤에서 로우의 선 자세와 같다. 정지 이미지의 정보 한계 | **어렵다.** 사람이 커져도 자세가 같다 |
| 바벨 로우 → 덤벨 로우 | 바벨(가는 선)이 작게 보여 덤벨과 구분이 어렵다(가설) | **가능성 있음.** 바벨이 더 잘 보이면 도움 |
| 바벨 로우 → 데드리프트 | 로우의 선 자세 프레임, Val 사람 고유의 특이 자세 | 일부만 |

- "사람이 작아서"와 "자세가 원래 같아서"는 **둘 다 원인이지만, 혼동 쌍마다 비중이 다릅니다.** 가장 큰 오류(데드리프트 → 로우)는 자세가 같은 쪽입니다.
- 이 결론은 Val 한 사람(Z23)의 관찰이고, 자세 분류(선 자세/숙인 자세)는 눈으로 본 정성 판단입니다. 오답 사진의 자세를 체계적으로 세는 데는 VLM을 가설 확인 재료로 쓸 수 있습니다(유료, 승인 필요).

**E07 개선 실험 후보(가설 하나에 변수 하나)**

1. **입력 해상도 올리기**(긴 변 224 → 384, 원본 가로 그대로): 바벨·덤벨 같은 가는 물체가 잘 보이게 한다. 기대 효과는 2↔9. 계산량은 약 2.9배(epoch 약 25초, 추정).
2. **클래스 가중치**(바벨 로우처럼 적은 클래스의 손실을 키움): Scratch + 증강에서 바벨 로우를 통째로 잃은 문제. 기대 효과는 9의 Recall.
3. (분석으로만 남김) 클립 단위 예측은 문제 정의(사진 한 장)를 바꾸므로 개선 실험으로 쓰지 않고, 한계와 향후 과제로 README에 적습니다.